# M4 Assignment2 - Green Patent Detection (PatentSBERTa): Active Learning + LLM→Human HITL
>Part C: Implement LLM → Human HITL (Gold Labels)

Suchanya Baiyam Business Data Science AAU


* Dataset:AI-Growth-Lab/patents_claims_1.5m_traim_test (The provided 50k samples)
* Model: AI-Growth-Lab/PatentSBERTa
* Working file: patents_50k_green.parquet with splits train_silver, pool_unlabeled, eval_silver
* Silver label: is_green_silver (derived from CPC Y02*)

Workflow (for each of the 100 rows):
- LLM step: given only the claim text, output: llm_green_suggested (0/1), llm_confidence (low/medium/high), and llm_rationale (1–3 sentences; cite phrases from the claim).
- Human step (final): review the claim + LLM output and set is_green_human (0/1).
- Optionally add notes (especially if you disagree with the LLM).

STEP01: Load libraries

In [ ]:
import pandas as pd
from IPython.display import clear_output

STEP02: Set up LLM and strutured prompt


STEP03: Load HITL 100 from part B


STEP04: LLM Evaluation Loop

**These steps has done locally using LLM Model: Ollama3:4b because of the quota of Gemini and OpenAi API**

In [ ]:
df = pd.read_csv("hitl_green_100_llm_evaluation.csv")

df["is_green_human"] = None
df["human_notes"] = None

df.to_csv("hitl_green_100_ready_for_human.csv", index=False)

STEP05: Human in the loop (review)

In [ ]:
if "is_green_human" not in df.columns:
    df["is_green_human"] = None
if "human_notes" not in df.columns:
    df["human_notes"] = None

print("Loaded:", len(df), "rows")

Loaded: 100 rows


STEP06: Override Analysis

In [ ]:
df = pd.read_csv("hitl_green_100_labeled.csv")

df["override"] = df["llm_green_suggested"] != df["is_green_human"]

override_count = df["override"].sum()

print("Number of overrides:", override_count)


In [ ]:
#code assisted by chatGPT to help make it quicker and more safe (after 10 times colab crashed)

for i, row in df.iterrows():

    # Skip already labeled rows (resume-safe)
    if pd.notnull(row["is_green_human"]):
        continue

    clear_output(wait=True)

    print("="*100)
    print(f"Reviewing index {i}  ({i+1}/{len(df)})")
    print("="*100)

    print("\n📄 CLAIM TEXT:\n")
    print(row["text"])

    print("\n🤖 LLM SUGGESTION:")
    print("Suggested:", row["llm_green_suggested"])
    print("Confidence:", row["llm_confidence"])
    print("Rationale:", row["llm_rationale"])

    print("\n------------------------------------")
    print("Press Enter to ACCEPT LLM suggestion")
    print("Or type 0 / 1 to override")
    print("------------------------------------")

    user_input = input("Your decision: ").strip()

    if user_input == "":
        final_label = row["llm_green_suggested"]
    else:
        final_label = int(user_input)

    note = input("Optional note (press Enter to skip): ")

    df.loc[i, "is_green_human"] = final_label
    df.loc[i, "human_notes"] = note

    # Save progress after every decision
    df.to_csv("hitl_green_100_human_progress.csv", index=False)

print("Human review complete!")

Reviewing index 99  (100/100)

📄 CLAIM TEXT:

1. A stainless steel having superior surface quality and moldability, comprising: in weight %, more than 0 to no more than 0.02% of C; more than 0 to no more than 0.02% of N; more than 0 to no more than 0.4% of Si; more than 0 to no more than 0.2% of Mn; more than 0 to no more than 0.04% of P; more than 0 to no more than 0.02% of S; 25.0 to 32.0% of Cr, 0 to 1.0% of Cu; more than 0 to no more than 0.8% of Ni; 0.01 to 0.5% of Ti; 0.01 to 0.05% of Nb, 0.01 to 1.5% of V; residual Fe; and inevitably contained elements, wherein the stainless steel meets Formula (1) below, and has yield point elongation of no more than 1.1%, and wherein the stainless steel further comprises (Ti,Nb) (C,N) precipitates, wherein an area fraction (%) of the entire precipitates per unit area in the stainless steel is no more than 3.5%, and an area fraction (%) of (Ti,Nb) (C,N) precipitates/entire precipitates is 62% or more.

🤖 LLM SUGGESTION:
Suggested: 0
Confidence:

STEP07: Count Override and Show 3 Override Examples

In [ ]:
df = pd.read_csv("hitl_green_100_human_progress.csv")

df["override"] = df["llm_green_suggested"] != df["is_green_human"]

override_count = df["override"].sum()

print("Total overrides:", override_count)
print("Override percentage:", round(override_count/len(df)*100, 2), "%")


Total overrides: 21
Override percentage: 21.0 %


In [ ]:
override_examples = df[df["override"] == True].head(3)

override_examples[[
    "text",
    "llm_green_suggested",
    "is_green_human",
    "llm_rationale",
    "human_notes"
]]

,text,llm_green_suggested,is_green_human,llm_rationale,human_notes
1,1. A standing pouch comprising: a first pack w...,1,0,This claim describes a standing pouch designed...,NaN
3,1. A method of detecting differences in densit...,1,0,This claim describes a method for detecting de...,NaN
9,"1. A welding wire feeding apparatus, comprisin...",1,0,This claim describes a welding wire feeding ap...,NaN


STEP08: Save override to CSV to work on other colab

In [ ]:
df.to_csv("gold_100_labeled.csv", index=False)